---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random, time
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

# Caută .env urcând din folderul curent până la root-ul repo-ului
ROOT = Path.cwd()

while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

load_dotenv(ROOT / ".env", override=True)

# Curățăm eventuale variabile OpenAI care pot interfera
for key in [
    "OPENAI_API_KEY",
    "OPENAI_ORG_ID",
    "OPENAI_PROJECT",
    "OPENAI_ORGANIZATION"
]:
    os.environ.pop(key, None)

# DeepSeek client
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

deepseek_client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)

DEEPSEEK_MODEL = "deepseek-chat"

# Gemini client, păstrat doar ca fallback opțional
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

gemini_client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"

# Alegem modelul principal pentru exercițiu
USE_GEMINI = False

client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL

print("Root proiect:", ROOT)
print("DeepSeek key found:", DEEPSEEK_API_KEY is not None)
print("Gemini key found:", GEMINI_API_KEY is not None)
print("Model folosit:", model_now)
print("OK")

Root proiect: c:\Users\osaci\Desktop\Proiect_Inginerie_AI\echochamber-project-team-1
DeepSeek key found: True
Gemini key found: True
Model folosit: deepseek-chat
OK


## Corpus

In [3]:
import pandas as pd
import random
from pathlib import Path

# găsim root-ul repo-ului
ROOT = Path.cwd()

while not (ROOT / ".git").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"

print("Corpus file:", corpus_file)
print("Exists:", corpus_file.exists())

corpus = pd.read_json(corpus_file, lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3, random_state=42).iterrows():
    print(f"[{c.get('source_channel', '')[:30]}] {c['text'][:80]}")

Corpus file: c:\Users\osaci\Desktop\Proiect_Inginerie_AI\echochamber-project-team-1\data\cleaned\corpus_youtube_large_clean.jsonl
Exists: True
30451 comentarii
Câmpuri: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']
[CălinGeorgescu-CanalulOficial] Multă sănătate dl. Președinte Călin Georgescu. Noi mergem până la capăt și vă do
[@CălinGeorgescu-CanalulOficial] Un discurs care trebuia sa vina de la Cotroceni! Multumim, d_le Calin Georgescu!
[RecorderRomania] Autoritatile abilitate sa intervina!!! De aceea sunt platiti de noi! Multumim re


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [4]:
# modifica dupa preferinte

AXA_1 = "anti_system_distrust"
AXA_2 = "cynical_moralizing_tone"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [5]:
AXA_1_DEFINITION = """
anti_system_distrust măsoară cât de mult comentariul exprimă neîncredere față de sistemul politic, instituții, partide, elite, presă, politicieni sau narațiuni oficiale.

0 = absent
Comentariul nu exprimă neîncredere anti-sistem. Poate fi neutru, descriptiv sau despre alt subiect.

1 = prezent
Comentariul exprimă scepticism, suspiciune, dezamăgire sau critică față de politicieni, instituții, presă sau sistem, dar nu într-un mod complet generalizat.

2 = dominant
Comentariul vede sistemul, instituțiile, elitele, partidele, presa sau politicienii ca profund compromise, corupte, false, ilegitime sau imposibil de reparat.
"""

AXA_2_DEFINITION = """
cynical_moralizing_tone măsoară cât de puternic este tonul cinic, acuzator, moralizator sau emoțional al comentariului.

0 = absent
Comentariul este calm, neutru sau descriptiv. Nu conține acuzații morale puternice, cinism sau furie vizibilă.

1 = prezent
Comentariul exprimă frustrare, dezamăgire, suspiciune, critică sau judecată morală moderată.

2 = dominant
Comentariul este puternic acuzator, furios, cinic, moralizator sau direct. Autorul sugerează că actorii politici, instituțiile sau sistemul sunt corupte, ipocrite, manipulate sau moral compromise.
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [6]:
MINI_PROMPT = f"""
Ești un analist de discurs politic.

SARCINĂ:
Adnotează comentariul politic folosind două axe:

1. {AXA_1}
2. {AXA_2}

CÂMPURI:
- target = ținta politică principală din comentariu
- stance = poziția față de target: pro / anti / neutru / ambiguu / none
- tone = modul dominant de formulare: calm / acuzator / moralizator / ironic / cinic / furios / suspicios / mobilizator / ambiguu
- {AXA_1} = 0 / 1 / 2
- {AXA_2} = 0 / 1 / 2

DEFINIȚII:
{AXA_1_DEFINITION}

{AXA_2_DEFINITION}

REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Nu cere modelului să identifice direct o „bulă discursivă”.
4. Dacă nu există target politic clar, folosește target="none" și stance="none".
5. Dacă textul este ironic sau sarcastic, codează sensul intenționat, nu doar sensul literal.
6. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
7. Separă emoția generală de poziția față de target.
8. Un comentariu poate fi furios sau cinic, dar totuși pro față de target dacă atacă adversarii targetului.
9. Returnează doar JSON valid. Nu adăuga explicații în afara JSON-ului.

FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0,
  "justification": ""
}}
"""

print(MINI_PROMPT)


Ești un analist de discurs politic.

SARCINĂ:
Adnotează comentariul politic folosind două axe:

1. anti_system_distrust
2. cynical_moralizing_tone

CÂMPURI:
- target = ținta politică principală din comentariu
- stance = poziția față de target: pro / anti / neutru / ambiguu / none
- tone = modul dominant de formulare: calm / acuzator / moralizator / ironic / cinic / furios / suspicios / mobilizator / ambiguu
- anti_system_distrust = 0 / 1 / 2
- cynical_moralizing_tone = 0 / 1 / 2

DEFINIȚII:

anti_system_distrust măsoară cât de mult comentariul exprimă neîncredere față de sistemul politic, instituții, partide, elite, presă, politicieni sau narațiuni oficiale.

0 = absent
Comentariul nu exprimă neîncredere anti-sistem. Poate fi neutru, descriptiv sau despre alt subiect.

1 = prezent
Comentariul exprimă scepticism, suspiciune, dezamăgire sau critică față de politicieni, instituții, presă sau sistem, dar nu într-un mod complet generalizat.

2 = dominant
Comentariul vede sistemul, instituț

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [7]:
TESTS = corpus.sample(5, random_state=12)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
114,yt_iH8jB4NlV9Y_UgyKDDUeih9qFHK-bMV4AaABAg,georgesimionoficial,Episodul 2: Cum ne-au furat alegerile - Turism...,Pentru a ii linisti pe toti cei care cred ca s...
7034,yt_HDsCdSOtxO0_Ugx7CCu0jecj244BqJN4AaABAg,RecorderRomania,EXPLICATIV RECORDER. Cum s-au repliat liderii ...,"Toti nemernicii astia odiosi, fara scrupule, c..."
8167,yt_VSvPoQLIj-Y_UgwN5XuNW0JHXGZ_mJx4AaABAg,RecorderRomania,Raiul evazioniștilor. Investigație din interio...,De ce tot ce este în acest documentar semăna i...
11538,yt_VDiv4TBODF8_UgxDI0t31RdfvkEQoFd4AaABAg,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,"Eu daca fur o paine si un pateu, pentru ca mor..."
4543,yt_bee6nXyzJ_E_Ugx8N2-6izdxWP-l8EJ4AaABAg,@CălinGeorgescu-CanalulOficial,Călin Georgescu împreună cu Anca Alexandrescu ...,"Din păcate există prea mulți "" spălați pe cree..."


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [8]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [9]:
import time

def llm(system, user, max_tokens=300, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client_now.chat.completions.create(
                model=model_now,
                temperature=0,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user}
                ]
            )
            return response.choices[0].message.content
        
        except Exception as e:
            error_name = type(e).__name__
            error_text = str(e)
            
            if "429" in error_text or "RateLimit" in error_name:
                wait_time = 60 * (attempt + 1)
                print(f"Rate limit atins. Aștept {wait_time} secunde...")
                time.sleep(wait_time)
            else:
                print("Eroare:", error_name)
                print(error_text[:500])
                raise e
    
    return """
{
  "target": "ERROR_rate_limit",
  "stance": "ambiguu",
  "tone": "ambiguu",
  "anti_system_distrust": 0,
  "cynical_moralizing_tone": 0,
  "justification": "Rate limit error: modelul nu a putut returna o adnotare după mai multe încercări."
}
"""

In [10]:
print("Model folosit:", model_now)
print("Client folosit:", client_now)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)

Model folosit: gemini-2.5-flash-lite
Client folosit: <openai.OpenAI object at 0x000001BCA3FF2850>
DeepSeek key: True
Gemini key: True


In [11]:
test_output = llm(
    "Returnează doar JSON valid.",
    "Adnotează comentariul: Guvernul este corupt și sistemul nu mai poate fi reparat."
)

print(test_output)

```json
{
  "comentariu": "Guvernul este corupt și sistemul nu mai poate fi reparat.",
  "adnotari": [
    {
      "text": "Guvernul este corupt",
      "entitate": "Guvern",
      "tip": "Instituție politică",
      "sentiment": "negativ",
      "subiectivitate": "ridicată"
    },
    {
      "text": "sistemul nu mai poate fi reparat",
      "entitate": "Sistem",
      "tip": "Sistem social/politic",
      "sentiment": "negativ",
      "subiectivitate": "ridicată"
    }
  ]
}
```


In [12]:
TESTS = corpus.dropna(subset=["text"]).sample(n=5, random_state=31).reset_index(drop=True)
TESTS

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_oYvauOG72Qs_UgyqL25mONe0cZaZCxF4AaABAg,youtube,turcescu111,Ar trebui sa fier lege sa nu poti candida la p...,Ar trebui sa fier lege sa nu poti candida la p...,NaN,False,NaN,NaN,oYvauOG72Qs,Psihiatria salvează România!,2026-03-06,2026-03-06,0,ro,2026-03-23
1,yt_VSvPoQLIj-Y_UgyKzrFjNnPCQO1IlpN4AaABAg,youtube,RecorderRomania,"Nu pot schimba sistemul astazi, dar pot sa dau...","Nu pot schimba sistemul astazi, dar pot sa dau...",NaN,False,NaN,NaN,VSvPoQLIj-Y,Raiul evazioniștilor. Investigație din interio...,2026-02-17,2026-02-19,0,ro,2026-03-22
2,yt_F3_3NlzeEpM_Ugz8vd0LpYWBS0H75kB4AaABAg,youtube,RecorderRomania,Avem de ales intre un Dobitoc anti occidenta s...,Avem de ales intre un Dobitoc anti occidenta s...,NaN,False,NaN,NaN,F3_3NlzeEpM,"50 de români, o dilemă: încotro ne îndreptăm?",2025-05-11,2025-05-16,0,ro,2026-03-22
3,yt_97qXZe3_J_k_Ugz_evtIRR6nEQv823l4AaABAg,youtube,turcescu111,"Domnule Turcescu, Zelenski cauta un prost de c...","Domnule Turcescu, Zelenski cauta un prost de c...",NaN,False,NaN,NaN,97qXZe3_J_k,Georgescu le-a dat la operație!,2026-03-16,2026-03-16,72,ro,2026-03-23
4,yt_0eaWJoXA4Y8_UgzZh7eTDqScy1ighZh4AaABAg,youtube,RecorderRomania,"Puteți să îmi spuneți și mie de ce ""fetele"" al...","Puteți să îmi spuneți și mie de ce ""fetele"" al...",NaN,False,NaN,NaN,0eaWJoXA4Y8,EXPLICATIV RECORDER: Cazul Gânj. Cum a devenit...,2025-10-23,2025-10-28,1,ro,2026-03-22


In [13]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Ar trebui sa fier lege sa nu poti candida la parlament fara studii superioare dovedite nu falsr E inacceptabil sa fii confid de invompetenti. Corporatiile de ce nu angajeaza cu 8 clase sau fiplome false.

OUTPUT MODEL:
```json
{
  "target": "clasa politică / politicieni",
  "stance": "anti",
  "tone": "acuzator",
  "anti_system_distrust": 1,
  "cynical_moralizing_tone": 2,
  "justification": "Comentariul critică aspru politicienii, sugerând că sunt incompetenți și că ar trebui să existe legi care să le impună studii superioare dovedite, nu false. Se face o comparație cu corporațiile, care nu angajează persoane cu diplome false, subliniind ipocrizia și incompetența din clasa politică. Există o neîncredere în sistemul politic actual, considerat a fi plin de incompetenți, iar tonul este acuzator și moralizator, sugerând că politicienii sunt corupți (prin diplome false) și inadecvați."
}
```
COMENTARIU:
Nu pot schimba sistemul astazi, dar pot sa dau documentarele voastre mai de

In [14]:
import json

cleaned = raw.replace("```json", "").replace("```", "").strip()

try:
    parsed = json.loads(cleaned)
    print("JSON valid: yes")
    print(parsed)
except Exception as e:
    print("JSON valid: no")
    print("Error:", e)
    print(cleaned)

JSON valid: yes
{'target': 'agresorul (Emil Gaje) și tipologia de bărbați similari', 'stance': 'anti', 'tone': 'acuzator', 'anti_system_distrust': 1, 'cynical_moralizing_tone': 2, 'justification': "Comentariul critică agresorul și bărbații cu istorii similare, sugerând că aceștia sunt 'monștri'. Există o neîncredere implicită în sistem, deoarece agresorul, un recidivist cu istoric de crimă, a putut să comită fapte noi. Tonul este puternic moralizator și acuzator, judecând atât agresorul, cât și, într-o oarecare măsură, victimele pentru alegerile lor, deși autorul menționează că nu vrea să dea vina pe victimă. Se exprimă dezgust și furie față de comportamentul agresorului și față de existența unor astfel de indivizi."}


## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce două axe ai ales?
Am ales axele 
anti_system_distrust și cynical_moralizing_tone. 
- De ce le-ai ales?
Am ales aceste axe deoarece profilul discursiv pe care îl analizez este orientat spre o perspectivă anti-sistem. Prima axă, **anti_system_distrust**, surprinde nivelul de neîncredere față de instituții, partide, elite, presă, politicieni sau sistemul politic în general. A doua axă, **cynical_moralizing_tone**, surprinde intensitatea tonului cinic, acuzator, moralizator sau emoțional al comentariului.

Împreună, cele două axe permit observarea modului în care comentariile combină neîncrederea instituțională cu un stil de exprimare critic, suspicios sau moralizator.

- Modelul a returnat JSON corect?
Modelul a returnat, în general, un JSON corect, cu cheile cerute în prompt. Structura răspunsului a fost clară și ușor de inspectat. Totuși, în unele cazuri, modelul a inclus răspunsul într-un bloc de tip `json`, ceea ce poate necesita curățarea textului înainte de parsare. Din punct de vedere al conținutului, unele clasificări pot fi discutabile, deoarece modelul tinde uneori să supra-interpreteze termenii asociați cu „sistemul” ca semne clare de discurs anti-sistem.
- Care a fost cea mai mare problemă?
O problemă observată este că modelul supra-interpretează uneori indiciile anti-sistem. De exemplu, simpla apariție a termenului „sistem” poate fi tratată ca dovadă de neîncredere anti-sistem, chiar dacă mesajul este mai degrabă moderat, reflexiv sau mobilizator. Prin urmare, promptul ar trebui să precizeze mai clar că o codare ridicată pentru anti_system_distrust necesită critici explicite la adresa instituțiilor, elitelor, partidelor, presei sau politicienilor ca fiind corupți, compromiși, falși sau ilegitimi.
- Ce ai schimba în prompt?
Ce aș schimba în prompt?

În urma testării, aș modifica promptul pentru a reduce riscul de supra-interpretare. Modelul a reușit în general să returneze un JSON structurat și să aplice cele două axe, dar uneori a tratat simpla apariție a unor termeni precum „sistem” ca dovadă suficientă pentru un cadru anti-sistem puternic. Din acest motiv, aș introduce o regulă mai strictă: scorul 2 pentru anti_system_distrust ar trebui acordat doar atunci când comentariul critică explicit instituțiile, partidele, elitele, presa sau politicienii ca fiind corupți, compromiși, falși sau ilegitimi.

Aș clarifica și diferența dintre tonul emoțional și poziționarea față de target. Un comentariu poate fi furios, cinic sau acuzator, dar totuși să fie pro față de un actor politic, dacă atacă adversarii acelui actor. De aceea, promptul ar trebui să insiste mai mult că stance-ul se referă strict la poziția față de target, nu la emoția generală a comentariului.

Aș mai adăuga o regulă pentru targeturile vagi. Dacă textul folosește termeni precum „ei”, „ăștia”, „sistemul” sau „hoții”, dar fără un referent clar, modelul ar trebui să marcheze targetul ca „unclear”, nu să inventeze o țintă politică prea precisă.

În final, aș cere modelului să returneze JSON brut, fără blocuri de tip ```json, și să limiteze justificarea la o singură propoziție scurtă, bazată strict pe text. Astfel, outputul ar fi mai ușor de parsat și mai puțin interpretativ.